In [1]:
import os
import json
import torch
import numpy as np
import pandas as pd
import timm
import librosa

# --- 1. CONFIGURATION ---
# Update these paths to where the files actually live on your laptop
MODEL_WEIGHTS = '../dataset/effnet_fold0_best.pth' 
LABEL_JSON    = '../dataset/label2idx.json' 
TEST_AUDIO    = '../dataset/test_soundscapes/my_test_soundscape.ogg' # The 1 file you want to test

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# --- 2. PREPROCESSING CONSTANTS ---
SR = 32000
N_SAMPLES = 5 * SR
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
F_MIN = 0
F_MAX = None
TOP_DB = 80.0

# --- 3. LOAD CLASSES ---
with open(LABEL_JSON, 'r') as f:
    label2idx = json.load(f)
    
# Reverse dictionary to get bird names from indices: {0: 'bird1', 1: 'bird2'}
idx2label = {v: k for k, v in label2idx.items()}
num_classes = len(idx2label)

# --- 4. LOAD MODEL ---
print("Loading model architecture and weights...")
model = timm.create_model(
    'tf_efficientnetv2_s.in21k_ft_in1k',
    pretrained=False, # We don't need internet, we are loading our own weights
    num_classes=num_classes,
    in_chans=1
)

checkpoint = torch.load(MODEL_WEIGHTS, map_location=DEVICE)
# # Handle Kaggle's nested 'model_state_dict' save format
# if 'model_state_dict' in checkpoint:
#     model.load_state_dict(checkpoint['model_state_dict'])
# else:
#     model.load_state_dict(checkpoint)

model.to(DEVICE)
model.eval() # CRITICAL: put model in evaluation mode
print("Model loaded successfully!\n")

# --- 5. PREPROCESSING FUNCTIONS ---
def normalize_waveform(y):
    """Normalize waveform to roughly [-1.0, 1.0]"""
    if len(y) == 0:
        return y
    return y / (np.max(np.abs(y)) + 1e-6)

def audio_to_melspec(waveform):
    """Convert waveform to normalized Mel Spectrogram image [0, 1]"""
    waveform = np.asarray(waveform, dtype=np.float32)
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel + TOP_DB) / TOP_DB
    log_mel = np.nan_to_num(log_mel, nan=0.0, posinf=1.0, neginf=0.0)
    log_mel = np.clip(log_mel, 0.0, 1.0)
    return log_mel.astype(np.float32)

# --- 6. INFERENCE ON 1 SOUNDSCAPE ---
print(f"Processing soundscape: {TEST_AUDIO}")

# Load the entire audio file
y, _ = librosa.load(TEST_AUDIO, sr=SR, mono=True)
duration = len(y) / SR

print(f"Total audio duration: {duration:.1f} seconds. Slicing into 5-second chunks...\n")

Using device: cpu
Loading model architecture and weights...
Model loaded successfully!

Processing soundscape: ../dataset/test_soundscapes/my_test_soundscape.ogg
Total audio duration: 60.0 seconds. Slicing into 5-second chunks...



In [2]:
# Load the exact structure expected by the competition
sample_sub = pd.read_csv(f'../dataset/sample_submission.csv')
EXPECTED_COLUMNS = sample_sub.columns.tolist()

with torch.no_grad():
    for end_sec in range(5, int(duration) + 1, 5):
        start_sec = end_sec - 5
        
        # 1. Slice the 5-second chunk
        start_idx = start_sec * SR
        end_idx = end_sec * SR
        chunk = y[start_idx:end_idx]
        
        # 2. Pad if chunk is somehow shorter than 5 seconds (usually happens at the very end of a file)
        if len(chunk) < N_SAMPLES:
            chunk = np.pad(chunk, (0, N_SAMPLES - len(chunk)))
            
        # 3. Preprocess exactly like training
        chunk = normalize_waveform(chunk)
        spec = audio_to_melspec(chunk)
        
        # 4. Convert to Tensor -> Add Batch (1) and Channel (1) dimensions -> shape: (1, 1, 128, 313)
        spec_tensor = torch.from_numpy(spec).unsqueeze(0).unsqueeze(0).to(DEVICE)
        
        # 5. Predict
        logits = model(spec_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
        
        # 6. View Results (Get Top 3 highest probability species for this chunk)
        top_3_indices = np.argsort(probs)[-3:][::-1]
        
        print(f"Time: {start_sec:02d}s - {end_sec:02d}s")
        for i in top_3_indices:
            print(f"   -> {idx2label[i]}: {probs[i]:.4f} ({probs[i]*100:.1f}%)")
        print("-" * 35)

print("\nInference complete!")

Time: 00s - 05s
   -> sobtyr1: 0.5047 (50.5%)
   -> ruther1: 0.5047 (50.5%)
   -> 1161364: 0.5045 (50.4%)
-----------------------------------
Time: 05s - 10s
   -> sobtyr1: 0.5047 (50.5%)
   -> ruther1: 0.5046 (50.5%)
   -> 1161364: 0.5044 (50.4%)
-----------------------------------
Time: 10s - 15s
   -> sobtyr1: 0.5050 (50.5%)
   -> ruther1: 0.5049 (50.5%)
   -> 1161364: 0.5047 (50.5%)
-----------------------------------
Time: 15s - 20s
   -> ruther1: 0.5056 (50.6%)
   -> sobtyr1: 0.5056 (50.6%)
   -> 1161364: 0.5052 (50.5%)
-----------------------------------
Time: 20s - 25s
   -> ruther1: 0.5055 (50.5%)
   -> sobtyr1: 0.5053 (50.5%)
   -> 1161364: 0.5051 (50.5%)
-----------------------------------
Time: 25s - 30s
   -> ruther1: 0.5055 (50.6%)
   -> sobtyr1: 0.5054 (50.5%)
   -> 1161364: 0.5051 (50.5%)
-----------------------------------
Time: 30s - 35s
   -> ruther1: 0.5052 (50.5%)
   -> sobtyr1: 0.5052 (50.5%)
   -> 1161364: 0.5049 (50.5%)
-----------------------------------
Time: 